# p = 1000: VIMCO-$\star$ training

Collapsed spike-and-slab regression with $\tau^2=1$. Run from this folder. Fits every tuning configuration for every dataset (seeds 0-49) and writes PIPs, logits and run metadata to `output_star_tau2_1`.

In [ ]:
%load_ext Cython

In [ ]:
%%cython --compile-args=-O3
# cython: language_level=3
# cython: boundscheck=False
# cython: wraparound=False
# cython: cdivision=True
# cython: initializedcheck=False

from pathlib import Path
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
start_all = time.time()
device = torch.device('cpu')
dtype = torch.float64
DATA_SEEDS = range(50)
METHOD_SEED = 12345
GRADIENT_IMPLEMENTATION = 'vimco_star_log_loo_v1'
tau2 = 1.0
prior_prec = 1.0 / tau2
jitter = 1e-08
a_pi = 0.02
b_pi = 1.0
TOTAL_SAMPLES = 300
S_GRID = [1]
if any((TOTAL_SAMPLES % int(S) != 0 for S in S_GRID)):
    raise ValueError('Every S in S_GRID must divide TOTAL_SAMPLES exactly.')
S_COPIES = int(S_GRID[0])
N_IMPORTANCE = TOTAL_SAMPLES // S_COPIES
LR_GRID = [0.01, 0.05, 0.1, 0.2, 0.3]
ALPHA_GRID = [0.2, 0.5, 0.7, 0.9]
ALPHA_START = 0.99
ALPHA_MIN = 0.0
ALPHA_STEP = 0.01
ESS_LOW = 0.8 * TOTAL_SAMPLES
PATIENCE_MAX = 200
BOUND_TOL = 0.0001
MAX_ITER = 3000
MAX_GRAD_NORM = 5.0
PROB_EPS = 1e-06
ETA_CLAMP = float(np.log((1.0 - PROB_EPS) / PROB_EPS))
data_dir = Path('data')
output_dir = Path('output_star_tau2_1')
output_dir.mkdir(parents=True, exist_ok=True)
LIKELIHOOD_ROUTE = 'adaptive'
ADAPTIVE_PRIMAL_MAX_Q = 50
PRIMAL_CHOL_BATCH_SIZE = 64
DUAL_CHOL_BATCH_SIZE = 8
VERBOSE_ITER = True
PRINT_EVERY = 100
NUM_TORCH_THREADS = 2
torch.set_num_threads(NUM_TORCH_THREADS)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass
true_model_file = data_dir / 'fixed_beta_and_active_set.csv'
if not true_model_file.exists():
    raise FileNotFoundError(f'Cannot find true model file: {true_model_file.resolve()}')
true_model_df = pd.read_csv(true_model_file)
required_true_cols = {'variable_name', 'is_intercept', 'is_true_active', 'beta'}
missing_true_cols = required_true_cols.difference(true_model_df.columns)
if missing_true_cols:
    raise ValueError(f'fixed_beta_and_active_set.csv is missing columns: {sorted(missing_true_cols)}')
if (len(true_model_df) != 1001
        or true_model_df['variable_name'].duplicated().any()
        or not true_model_df['is_intercept'].isin([0, 1]).all()):
    raise ValueError('The truth file must have 1001 distinct variables and binary is_intercept flags.')

slope_model_df = true_model_df.loc[true_model_df['is_intercept'].astype(int) == 0].copy().reset_index(drop=True)
p = int(len(slope_model_df))
feature_cols = [f'X{j}' for j in range(1, p + 1)]

if p != 1000 or slope_model_df['variable_name'].duplicated().any():
    raise ValueError('This notebook requires exactly 1000 distinct slope variables.')
if set(slope_model_df['variable_name']) != set(feature_cols):
    raise ValueError('Slope variables must be named X1 through X1000.')
slope_model_df = slope_model_df.set_index('variable_name').loc[feature_cols].reset_index()
if true_model_df['is_intercept'].astype(int).eq(1).sum() != 1:
    raise ValueError('The truth file must contain exactly one intercept.')
if not np.isfinite(true_model_df['beta'].to_numpy(dtype=np.float64)).all():
    raise ValueError('True coefficients must be finite.')
if (not slope_model_df['is_true_active'].isin([0, 1]).all()
        or not np.array_equal(slope_model_df['is_true_active'].to_numpy(dtype=int),
                              (slope_model_df['beta'].to_numpy(dtype=float) != 0).astype(int))):
    raise ValueError('True inclusion indicators must match the nonzero slope coefficients.')
truth_np = slope_model_df['is_true_active'].to_numpy(dtype=np.float64)
beta_slopes_np = slope_model_df['beta'].to_numpy(dtype=np.float64)
log_2pi_t = torch.tensor(float(np.log(2.0 * np.pi)), dtype=dtype, device=device)
tau2_t = torch.tensor(tau2, dtype=dtype, device=device)
prior_prec_t = torch.tensor(prior_prec, dtype=dtype, device=device)
log_prior_prec_t = torch.log(prior_prec_t)
a_pi_t = torch.tensor(a_pi, dtype=dtype, device=device)
b_pi_t = torch.tensor(b_pi, dtype=dtype, device=device)
p_ab_t = torch.tensor(float(p + a_pi + b_pi), dtype=dtype, device=device)
LOG_BETA_AB_t = torch.lgamma(a_pi_t) + torch.lgamma(b_pi_t) - torch.lgamma(a_pi_t + b_pi_t)
_MODEL_SIZE_GRID_t = torch.arange(p + 1, dtype=dtype, device=device)
_LOG_MODEL_PRIOR_BY_SIZE_t = torch.lgamma(a_pi_t + _MODEL_SIZE_GRID_t) + torch.lgamma(b_pi_t + p - _MODEL_SIZE_GRID_t) - torch.lgamma(p_ab_t) - LOG_BETA_AB_t
n = None
nu_t = None
log_n_t = None
marginal_constant_t = None
XtX_t = None
Xty_t = None
yty_t = None
Xc_t = None
Xc_T_t = None
yc_t = None
ycyc_t = None
I_n_t = None
_REG_CACHE = {}


def leave_one_out_logsumexp(values, dim=0):
    count = int(values.shape[dim])
    if count < 2:
        raise ValueError('Leave-one-out baselines require at least two importance samples.')
    prefix = torch.logcumsumexp(values, dim=dim)
    suffix = torch.flip(torch.logcumsumexp(torch.flip(values, dims=(dim,)), dim=dim), dims=(dim,))
    pad = torch.full_like(values.narrow(dim, 0, 1), -float('inf'))
    left = torch.cat((pad, prefix.narrow(dim, 0, count - 1)), dim=dim)
    right = torch.cat((suffix.narrow(dim, 1, count - 1), pad), dim=dim)
    return torch.logaddexp(left, right)


def regularisation_cached(q):
    q = int(q)
    mat = _REG_CACHE.get(q)
    if mat is None:
        prior_diag = torch.full((q,), prior_prec, dtype=dtype, device=device)
        prior_diag[0] = 0.0
        mat = torch.diag(prior_diag) + jitter * torch.eye(q, dtype=dtype, device=device)
        _REG_CACHE[q] = mat
    return mat

def set_dataset_globals(X_full_np, X_slope_np, y_np):
    global n, XtX_t, Xty_t, yty_t
    global Xc_t, Xc_T_t, yc_t, ycyc_t, I_n_t
    global nu_t, log_n_t, marginal_constant_t, _REG_CACHE
    n = int(X_full_np.shape[0])
    _REG_CACHE = {}
    X_full_np = np.ascontiguousarray(X_full_np, dtype=np.float64)
    X_slope_np = np.ascontiguousarray(X_slope_np, dtype=np.float64)
    y_np = np.ascontiguousarray(y_np, dtype=np.float64)
    XtX_np = np.asarray(X_full_np.T @ X_full_np, dtype=np.float64)
    Xty_np = np.asarray(X_full_np.T @ y_np, dtype=np.float64)
    yty_np = float(y_np @ y_np)
    XtX_t = torch.from_numpy(XtX_np).to(device=device, dtype=dtype)
    Xty_t = torch.from_numpy(Xty_np).to(device=device, dtype=dtype)
    yty_t = torch.tensor(yty_np, dtype=dtype, device=device)
    Xc_np = X_slope_np - X_slope_np.mean(axis=0, keepdims=True)
    yc_np = y_np - y_np.mean()
    Xc_np = np.ascontiguousarray(Xc_np, dtype=np.float64)
    yc_np = np.ascontiguousarray(yc_np, dtype=np.float64)
    Xc_t = torch.from_numpy(Xc_np).to(device=device, dtype=dtype)
    Xc_T_t = Xc_t.t().contiguous()
    yc_t = torch.from_numpy(yc_np).to(device=device, dtype=dtype)
    ycyc_t = torch.sum(yc_t.square())
    I_n_t = torch.eye(n, dtype=dtype, device=device)
    nu_t = torch.tensor(float(n - 1), dtype=dtype, device=device)
    log_n_t = torch.log(torch.tensor(float(n), dtype=dtype, device=device))
    marginal_constant_t = torch.lgamma(0.5 * nu_t) - 0.5 * nu_t * log_2pi_t

def load_dataset(DATA_SEED):
    dataset_file = data_dir / f'seed_{DATA_SEED:02d}.csv'
    if not dataset_file.exists():
        raise FileNotFoundError(f'Cannot find dataset: {dataset_file.resolve()}')
    dataset_df = pd.read_csv(dataset_file)
    required_cols = ['Intercept'] + feature_cols + ['y']
    missing_cols = [c for c in required_cols if c not in dataset_df.columns]
    if missing_cols:
        raise ValueError(f'{dataset_file.name} is missing columns: {missing_cols[:10]}')
    X_full_np = dataset_df[['Intercept'] + feature_cols].to_numpy(dtype=np.float64)
    X_slope_np = dataset_df[feature_cols].to_numpy(dtype=np.float64)
    y_np = dataset_df['y'].to_numpy(dtype=np.float64)

    if len(y_np) < 3 or not np.isfinite(X_full_np).all() or not np.isfinite(y_np).all():
        raise ValueError(f'{dataset_file.name}: need at least three finite observations.')
    if not np.allclose(X_full_np[:, 0], 1.0, rtol=0, atol=1e-12):
        raise ValueError(f'{dataset_file.name}: Intercept must equal one.')
    if float(np.var(y_np)) <= 0.0:
        raise ValueError(f'{dataset_file.name}: y must have positive variance.')
    set_dataset_globals(X_full_np, X_slope_np, y_np)
    return dataset_file

def clamp_eta_(eta_tensor):
    with torch.no_grad():
        eta_tensor.clamp_(min=-ETA_CLAMP, max=ETA_CLAMP)

def gamma_from_eta(eta):
    eta_safe = torch.clamp(eta, min=-ETA_CLAMP, max=ETA_CLAMP)
    return torch.sigmoid(eta_safe)

def sample_gamma(gamma, total_samples):
    return torch.bernoulli(gamma.expand(total_samples, -1))

def log_q_eta(G, eta):
    eta_safe = torch.clamp(eta, min=-ETA_CLAMP, max=ETA_CLAMP)
    return (G * eta_safe[None, :] - F.softplus(eta_safe)[None, :]).sum(dim=1)

def make_initial_eta():
    gamma_init = a_pi / (a_pi + b_pi)
    gamma_init = min(max(float(gamma_init), PROB_EPS), 1.0 - PROB_EPS)
    eta_value = float(np.log(gamma_init / (1.0 - gamma_init)))
    eta_init = torch.full((p,), eta_value, dtype=dtype, device=device)
    eta_init.clamp_(min=-ETA_CLAMP, max=ETA_CLAMP)
    return eta_init

def log_model_prior_torch(G):
    model_size = G.sum(dim=1).to(torch.long)
    return _LOG_MODEL_PRIOR_BY_SIZE_t[model_size]

def log_marginal_likelihood_primal_torch(G):
    number_models = int(G.shape[0])
    out = torch.empty(number_models, dtype=dtype, device=device)
    active_counts = G.sum(dim=1).to(torch.long)
    unique_counts = torch.unique(active_counts)
    for count_tensor in unique_counts:
        k_slope = int(count_tensor.item())
        row_indices_all = torch.nonzero(active_counts == k_slope, as_tuple=False).flatten()
        q = k_slope + 1
        for start in range(0, int(row_indices_all.numel()), PRIMAL_CHOL_BATCH_SIZE):
            row_indices = row_indices_all[start:start + PRIMAL_CHOL_BATCH_SIZE]
            batch_size = int(row_indices.numel())
            if k_slope == 0:
                idx_mat = torch.zeros((batch_size, 1), dtype=torch.long, device=device)
            else:
                group_G = G.index_select(0, row_indices)
                nonzero_locations = torch.nonzero(group_G > 0.5, as_tuple=False)
                slope_idx = nonzero_locations[:, 1].reshape(batch_size, k_slope).to(torch.long) + 1
                intercept = torch.zeros((batch_size, 1), dtype=torch.long, device=device)
                idx_mat = torch.cat((intercept, slope_idx), dim=1)
            XtX_batch = XtX_t[idx_mat[:, :, None], idx_mat[:, None, :]]
            Xty_batch = Xty_t[idx_mat]
            M_batch = XtX_batch + regularisation_cached(q).unsqueeze(0)
            L_batch = torch.linalg.cholesky(M_batch)
            solve_batch = torch.cholesky_solve(Xty_batch[..., None], L_batch).squeeze(-1)
            residual_batch = yty_t - torch.sum(Xty_batch * solve_batch, dim=1)
            residual_batch = torch.clamp(residual_batch, min=jitter)
            logdet_M_batch = 2.0 * torch.sum(torch.log(torch.diagonal(L_batch, dim1=-2, dim2=-1)), dim=1)
            values = marginal_constant_t + 0.5 * k_slope * log_prior_prec_t - 0.5 * logdet_M_batch - 0.5 * nu_t * torch.log(residual_batch / 2.0)
            out[row_indices] = values
    return out

def log_marginal_likelihood_dual_torch(G):
    number_models = int(G.shape[0])
    out = torch.empty(number_models, dtype=dtype, device=device)
    active_counts = G.sum(dim=1).to(torch.long)
    unique_counts = torch.unique(active_counts)
    for count_tensor in unique_counts:
        k_slope = int(count_tensor.item())
        row_indices = torch.nonzero(active_counts == k_slope, as_tuple=False).flatten()
        group_size = int(row_indices.numel())
        if k_slope == 0:
            residual = torch.clamp(ycyc_t, min=jitter)
            value = marginal_constant_t - 0.5 * log_n_t - 0.5 * nu_t * torch.log(residual / 2.0)
            out[row_indices] = value.expand(group_size)
            continue
        group_G = G.index_select(0, row_indices)
        nonzero_locations = torch.nonzero(group_G > 0.5, as_tuple=False)
        slope_idx = nonzero_locations[:, 1].reshape(group_size, k_slope).to(torch.long)
        for start in range(0, group_size, DUAL_CHOL_BATCH_SIZE):
            end = min(start + DUAL_CHOL_BATCH_SIZE, group_size)
            current_rows = row_indices[start:end]
            current_idx = slope_idx[start:end]
            current_batch_size = int(current_idx.shape[0])
            XA = Xc_T_t[current_idx].transpose(1, 2).contiguous()
            K_batch = I_n_t.unsqueeze(0) + tau2_t * torch.bmm(XA, XA.transpose(1, 2))
            if jitter > 0.0:
                K_batch = K_batch + jitter * I_n_t.unsqueeze(0)
            L_batch = torch.linalg.cholesky(K_batch)
            y_batch = yc_t.expand(current_batch_size, -1).unsqueeze(2)
            solved = torch.cholesky_solve(y_batch, L_batch).squeeze(2)
            residual_batch = torch.sum(yc_t.unsqueeze(0) * solved, dim=1)
            residual_batch = torch.clamp(residual_batch, min=jitter)
            logdet_K_batch = 2.0 * torch.sum(torch.log(torch.diagonal(L_batch, dim1=-2, dim2=-1)), dim=1)
            values = marginal_constant_t - 0.5 * log_n_t - 0.5 * logdet_K_batch - 0.5 * nu_t * torch.log(residual_batch / 2.0)
            out[current_rows] = values
    return out

def log_marginal_likelihood_adaptive_torch(G):
    active_counts = G.sum(dim=1).to(torch.long)
    primal_mask = active_counts + 1 <= int(ADAPTIVE_PRIMAL_MAX_Q)
    out = torch.empty(int(G.shape[0]), dtype=dtype, device=device)
    if bool(primal_mask.any().item()):
        idx = torch.nonzero(primal_mask, as_tuple=False).flatten()
        out[idx] = log_marginal_likelihood_primal_torch(G.index_select(0, idx))
    if bool((~primal_mask).any().item()):
        idx = torch.nonzero(~primal_mask, as_tuple=False).flatten()
        out[idx] = log_marginal_likelihood_dual_torch(G.index_select(0, idx))
    return out

def log_marginal_likelihood_torch(G):
    if LIKELIHOOD_ROUTE == 'primal':
        return log_marginal_likelihood_primal_torch(G)
    if LIKELIHOOD_ROUTE == 'dual':
        return log_marginal_likelihood_dual_torch(G)
    if LIKELIHOOD_ROUTE == 'adaptive':
        return log_marginal_likelihood_adaptive_torch(G)
    raise ValueError("LIKELIHOOD_ROUTE must be 'primal', 'dual', or 'adaptive'.")

def log_joint_integrated_torch(G):
    return log_marginal_likelihood_torch(G) + log_model_prior_torch(G)

def clean_float(x):
    return str(x).replace('.', 'p').replace('-', 'm')

def vimco_star_grad_eta(eta, total_samples, S_copies, alpha, f_prev_scaled, f_prev_shift):
    one_minus_alpha = max(1e-12, 1.0 - float(alpha))
    N_importance = total_samples // S_copies
    gamma = gamma_from_eta(eta)
    G_total = sample_gamma(gamma, total_samples)
    logp_total = log_joint_integrated_torch(G_total)
    logq_total = log_q_eta(G_total, eta)
    logw_total = logp_total - logq_total
    if not bool(torch.isfinite(logw_total).all().item()):
        raise FloatingPointError('Non-finite VIMCO* log importance weight detected.')
    f_new_scaled, f_new_shift = estimate_star_f_scaled_from_batch(G_total, logw_total, gamma, alpha)
    G_blocks = G_total.reshape(S_copies, N_importance, p)
    logw_blocks = logw_total.reshape(S_copies, N_importance)
    grad_sum = torch.zeros(p, dtype=dtype, device=device)
    bound_sum = torch.tensor(0.0, dtype=dtype, device=device)
    ess_sum = torch.tensor(0.0, dtype=dtype, device=device)
    for r in range(S_copies):
        G = G_blocks[r]
        logw = logw_blocks[r]
        logu_abs = one_minus_alpha * logw
        current_shift = torch.max(logu_abs)
        u_scaled = torch.exp(logu_abs - current_shift)
        sum_u_scaled = torch.sum(u_scaled)
        wbar = u_scaled / sum_u_scaled
        ess_sum = ess_sum + 1.0 / torch.sum(wbar.square())
        score = G - gamma[None, :]
        term1 = -torch.sum(wbar[:, None] * score, dim=0)
        if f_prev_scaled is None:
            log_f_loo = estimate_star_f_loo_log(G, logu_abs, alpha)
            log_other = leave_one_out_logsumexp(logu_abs)[:, None]
            log_inside = (torch.logaddexp(log_other, log_f_loo)
                          - torch.logsumexp(logu_abs, dim=0))
        else:
            log_inside = lagged_log_inside(logu_abs, f_prev_scaled, f_prev_shift)
        term2 = -torch.sum(score * log_inside, dim=0) / one_minus_alpha
        grad_sum += term1 + term2
        bound_r = (current_shift + torch.log(sum_u_scaled) - np.log(N_importance)) / one_minus_alpha
        bound_sum += bound_r
    grad = grad_sum / S_copies
    bound = bound_sum / S_copies
    if not bool(torch.isfinite(grad).all().item()):
        raise FloatingPointError('Non-finite VIMCO* gradient detected.')
    return (grad, f_new_scaled.detach(), float(f_new_shift), float(bound.item()), float(ess_sum.item()))

def run_vimco_star(lr, alpha=None, annealed=False):
    method_start = time.time()
    torch.manual_seed(METHOD_SEED)
    np.random.seed(METHOD_SEED)
    eta = torch.nn.Parameter(make_initial_eta())
    clamp_eta_(eta)
    optimizer = torch.optim.Adam([eta], lr=lr, betas=(0.9, 0.999))
    current_alpha = float(ALPHA_START if annealed else alpha)
    bound_hist = []
    alpha_hist = []
    ess_hist = []
    f_prev_scaled = None
    f_prev_shift = None
    best_training_bound = -float('inf')
    best_eta = None
    best_iter = 0
    patience = 0
    stop_reason = None
    for it in range(1, MAX_ITER + 1):
        optimizer.zero_grad(set_to_none=True)
        alpha_used = float(current_alpha)
        with torch.no_grad():
            eta_before = eta.detach().clone()
            grad, f_new_scaled, f_new_shift, training_bound, ess_value = vimco_star_grad_eta(eta, TOTAL_SAMPLES, S_COPIES, alpha_used, f_prev_scaled, f_prev_shift)
        bound_hist.append(training_bound)
        alpha_hist.append(alpha_used)
        ess_hist.append(ess_value)
        if training_bound > best_training_bound + BOUND_TOL:
            best_training_bound = training_bound
            best_eta = eta_before.clone()
            best_iter = it
            patience = 0
        else:
            if training_bound > best_training_bound:
                best_training_bound = training_bound
                best_eta = eta_before.clone()
                best_iter = it
            patience += 1
        eta.grad = (-grad).clone()
        torch.nn.utils.clip_grad_norm_([eta], max_norm=MAX_GRAD_NORM)
        optimizer.step()
        clamp_eta_(eta)
        f_prev_scaled = f_new_scaled.clone()
        f_prev_shift = float(f_new_shift)
        if annealed and ess_value >= ESS_LOW and (it < MAX_ITER):
            alpha_candidate = max(ALPHA_MIN, alpha_used - ALPHA_STEP)
            if alpha_candidate < alpha_used - 1e-12:
                current_alpha = float(alpha_candidate)
                f_prev_scaled = None
                f_prev_shift = None
                best_training_bound = -float('inf')
                best_eta = None
                best_iter = 0
                patience = 0
            else:
                current_alpha = alpha_used
        else:
            current_alpha = alpha_used
        if VERBOSE_ITER and it % PRINT_EVERY == 0:
            print(f'VIMCO* iter {it:5d} | lr={lr:.3g} | alpha={alpha_used:.2f} | bound={training_bound:.3f} | ESS={ess_value:.1f} | patience={patience}/{PATIENCE_MAX}')
        if patience >= PATIENCE_MAX:
            stop_reason = f'patience reached {PATIENCE_MAX}'
            break
    else:
        it = MAX_ITER
        stop_reason = f'maximum iterations reached {MAX_ITER}'
    last_eta = eta.detach().clone()
    training_state = {'last_eta': last_eta, 'optimizer': optimizer.state_dict(), 'torch_rng_state': torch.get_rng_state(), 'numpy_rng_state': np.random.get_state(), 'patience': patience}
    training_state['current_alpha'] = current_alpha
    training_state['f_prev_scaled'] = f_prev_scaled
    training_state['f_prev_shift'] = f_prev_shift
    if best_eta is not None:
        with torch.no_grad():
            eta.copy_(best_eta)
            clamp_eta_(eta)
    pip = gamma_from_eta(eta).detach().cpu().numpy()
    eta_np = eta.detach().cpu().numpy()
    return {'training_state': training_state, 'pip': pip, 'lambda': eta_np, 'bound': np.asarray(bound_hist), 'alpha_history': np.asarray(alpha_hist), 'ess_history': np.asarray(ess_hist), 'time': time.time() - method_start, 'iters': it, 'best_iter': best_iter, 'best_training_bound': best_training_bound, 'method_seed': METHOD_SEED, 'saved_alpha': float(alpha_hist[best_iter - 1]) if best_iter > 0 else float(alpha_hist[len(alpha_hist) - 1]), 'initial_alpha': float(ALPHA_START if annealed else alpha), 'final_alpha': float(alpha_hist[len(alpha_hist) - 1]) if len(alpha_hist) > 0 else np.nan, 'stop': stop_reason}


def estimate_star_f_scaled_from_batch(G, logw, gamma, alpha):
    logu = (1.0 - float(alpha)) * logw
    shift = torch.max(logu)
    u = torch.exp(logu - shift)
    # For Bernoulli scores, both weighted and unweighted variances are
    # invariant to subtracting gamma. Compute them from inclusion counts.
    frequency = G.mean(dim=0)
    variance = frequency * (1.0 - frequency)
    mass_one = (u[:, None] * G).sum(dim=0)
    mass_zero = (u[:, None] * (1.0 - G)).sum(dim=0)
    safe_variance = torch.where(variance > 0, variance, torch.ones_like(variance))
    f_scaled = float(alpha) * (mass_one / u.sum()) * (mass_zero / len(G)) / safe_variance
    f_scaled = torch.where(variance > 0, f_scaled, torch.zeros_like(f_scaled))
    return (f_scaled.detach(), float(shift.item()))


def estimate_star_f_loo_log(G, logu, alpha):
    n_loo = int(G.shape[0]) - 1
    if n_loo < 1:
        raise ValueError('VIMCO* requires N > 1.')
    if alpha <= 0.0:
        return torch.full_like(G, -float('inf'))
    minus_inf = torch.full_like(G, -float('inf'))
    log_mass_one = leave_one_out_logsumexp(torch.where(G > 0.5, logu[:, None], minus_inf))
    log_mass_zero = leave_one_out_logsumexp(torch.where(G < 0.5, logu[:, None], minus_inf))
    log_mass = torch.logaddexp(log_mass_one, log_mass_zero)
    count_one = G.sum(dim=0, keepdim=True) - G
    count_zero = float(n_loo) - count_one
    varying = (count_one > 0) & (count_zero > 0)
    safe_one = torch.where(varying, count_one, torch.ones_like(count_one))
    safe_zero = torch.where(varying, count_zero, torch.ones_like(count_zero))
    # This is the original optimal-baseline moment ratio, evaluated in log
    # space. With zero sample variance choose the valid zero baseline;
    # adding eps to a 0/0 ratio would create a spurious baseline.
    log_f = (np.log(float(alpha)) + np.log(float(n_loo))
             + log_mass_one + log_mass_zero - log_mass
             - torch.log(safe_one) - torch.log(safe_zero))
    return torch.where(varying, log_f, minus_inf).detach()


def lagged_log_inside(logu, f_prev_scaled, f_prev_shift):
    log_other = leave_one_out_logsumexp(logu)[:, None]
    log_f = torch.where(f_prev_scaled > 0, torch.log(f_prev_scaled),
                        torch.full_like(f_prev_scaled, -float('inf')))
    log_f = log_f + float(f_prev_shift)
    return torch.logaddexp(log_other, log_f[None, :]) - torch.logsumexp(logu, dim=0)
METHOD_NAME = 'VIMCO*'
METHOD_KEY = 'vimco_star'
METHOD_IS_ANNEALED = False

import hashlib

def atomic_csv(frame, path):
    temporary = path.with_suffix(".tmp")
    frame.to_csv(temporary, index=False, float_format="%.17g")
    temporary.replace(path)

def settings():
    for S in S_GRID:
        N = TOTAL_SAMPLES // S
        for alpha in ALPHA_GRID:
            for lr in LR_GRID:
                yield lr, alpha, S, N, f"star_alpha_{clean_float(alpha)}_lr_{clean_float(lr)}_S_{S}_N_{N}"


def run_training():
    global S_COPIES, N_IMPORTANCE
    all_rows = []
    model_hash = hashlib.sha256(true_model_file.read_bytes()).hexdigest()
    for data_seed in DATA_SEEDS:
        dataset_file = load_dataset(data_seed)
        dataset_hash = hashlib.sha256(dataset_file.read_bytes()).hexdigest()
        pip_file = output_dir / f"pips_seed_{data_seed:02d}.csv"
        runtime_file = output_dir / f"runtime_and_metrics_seed_{data_seed:02d}.csv"
        pips = {"variable_name": feature_cols, "is_true_active": truth_np.astype(int), "beta": beta_slopes_np}
        rows = []
        for lr, alpha, S, N, label in settings():
            S_COPIES, N_IMPORTANCE = S, N
            print(f"Training seed {data_seed:02d} | {label}", flush=True)
            if METHOD_KEY == "elbo":
                out = run_vimco_star(lr=lr)
            else:
                out = run_vimco_star(lr=lr, alpha=alpha, annealed=METHOD_IS_ANNEALED)
            pip_col, lambda_col = f"pip_{label}", f"lambda_{label}"
            pips[pip_col], pips[lambda_col] = out["pip"], out["lambda"]
            rows.append({
                "seed": int(data_seed), "method": METHOD_NAME, "S": S, "N": N,
                "learning_rate": lr,
                "alpha_setting": "annealed" if METHOD_IS_ANNEALED else alpha,
                "initial_alpha": out.get("initial_alpha", np.nan),
                "final_alpha": out.get("final_alpha", np.nan),
                "saved_alpha": out.get("saved_alpha", np.nan),
                "iters": out["iters"], "best_iter": out["best_iter"],
                "training_bound_saved": out["best_training_bound"],
                "method_seed": METHOD_SEED, "tau2": tau2, "a_pi": a_pi, "b_pi": b_pi,
                "dataset_sha256": dataset_hash, "true_model_sha256": model_hash,
                "n": int(n), "p": int(p),
                "gradient_implementation": GRADIENT_IMPLEMENTATION,
                "likelihood_route": LIKELIHOOD_ROUTE,
                "adaptive_primal_max_q": ADAPTIVE_PRIMAL_MAX_Q,
                "time (s)": out["time"], "stop": out["stop"],
                "pip_file": str(pip_file), "pip_column": pip_col,
                "lambda_column": lambda_col,
            })
        atomic_csv(pd.DataFrame(pips), pip_file)
        atomic_csv(pd.DataFrame(rows), runtime_file)
        all_rows.extend(rows)
    atomic_csv(pd.DataFrame(all_rows), output_dir / "runtime_and_metrics_all_seeds.csv")
    print("Training complete. Total wall time:", time.time() - start_all)

run_training()
